# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 12** you build `fp.Design` **binder** objects from both binder pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1). The binder is the recognition element of the
sensor — it must pass the binder bar *before* you couple it to a switch.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `Design` (binder) objects from the pools

Map each binder-pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the
layers: `scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and
`rosetta_dG`/`shape_complementarity`/`solubility` (Layer 3 physics). We keep `paradigm` +
`epitope_coverage` in `extra` for the integration + ON/OFF analysis in notebook 04. (Mock has no
independent orthogonal predictor, so we run Layers 1+3 here; on Colab add a second predictor for
Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the binder pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import biosensor_tools as bt
    ANALYTE, HOTSPOTS = "ANALYTE", bt.parse_hotspots("A12,A45,A60")
    bc = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   epitope_coverage=bt.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "epitope_coverage": r.get("epitope_coverage")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each paradigm separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [ ]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the
**combined** pool for one comparable figure; the per-paradigm runs above are the rigorous version.
Read the bars as a funnel: steep drops show which layer discriminates.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p12")
print("\nsaved results/p12_survival.png + results/p12_ranked.csv")
top

## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number that decides
which binders are worth coupling to a switch in notebook 04. Remember: survival is *enrichment*, not
*correctness*, and even a passing binder is only the *recognition* half of a sensor. Mock numbers are
SYNTHETIC.

In [ ]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p12_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — integrate binder + switch and reason about ON/OFF states.